# Phase 1 (définitive) — Séparer les instruments par NMF INFORMÉE

Leçon du notebook précédent : classer chaque frappe du mix brut **ne marche pas** (chevauchement + décalage de domaine → le dunun disparaît). La solution validée : **séparer d'abord**.

**NMF informée** (= la séparation des thèses, mais guidée par notre vérité terrain) :
1. On apprend un **dictionnaire spectral** du dunun et du jembé sur les **pistes isolées** du corpus.
2. On l'applique au mix : on garde le dictionnaire **fixe** et on résout seulement *quand* chaque voix joue.
3. On obtient l'**activité dunun** et l'**activité jembé** séparées → onsets, stems ré-écoutables, carte.

Résultat (validé plus bas) : détection du dunun sur mix synthétique **F≈0,99** (vs 0,70 pour un simple filtre de bande), et sur le vrai YouTube un **pouls de dunun** crédible.

## 0. Imports, config & fonctions

In [ ]:
import os, glob, warnings
import numpy as np, pandas as pd, librosa
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from sklearn.decomposition import NMF
from scipy.signal import butter, sosfiltfilt
warnings.filterwarnings('ignore')

SR, NFFT, HOP = 22050, 2048, 512
CORP = 'corpus/MJ_Woloso'
OUT  = 'output'; os.makedirs(OUT, exist_ok=True)
KD, KJ = 6, 12   # nb de briques spectrales pour dunun / jembé

def load(f):  return librosa.load(f, sr=SR, mono=True)[0]
def spec(y):  return np.abs(librosa.stft(y, n_fft=NFFT, hop_length=HOP))
freqs = librosa.fft_frequencies(sr=SR, n_fft=NFFT)

def learn_dict(files, k):
    """Apprend un dictionnaire spectral (freq x k) sur des pistes isolées."""
    S = np.concatenate([spec(load(f)) for f in files if os.path.exists(f)], axis=1)
    m = NMF(n_components=k, init='nndsvda', max_iter=200, random_state=0)
    m.fit(S.T)
    return m.components_.T   # freq x k

def activations(S, W, iters=100):
    """Dictionnaire W FIXE -> résout les activations H (k x temps) sur S."""
    H = np.random.RandomState(0).rand(W.shape[1], S.shape[1]) + 1e-3
    WtW = W.T @ W
    for _ in range(iters):
        H *= (W.T @ S) / (WtW @ H + 1e-9)
    return H

def fmeasure(pred, gt, tol=0.05):
    pred = sorted(pred); gt = sorted(map(float, gt)); used = [False]*len(gt); tp = 0
    for p in pred:
        best, bd = -1, tol
        for i, g in enumerate(gt):
            if used[i]: continue
            d = abs(p-g)
            if d <= bd: bd, best = d, i
        if best >= 0: used[best] = True; tp += 1
    P = tp/len(pred) if len(pred) else 0
    R = tp/len(gt) if len(gt) else 0
    return P, R, (2*P*R/(P+R) if P+R else 0)

print('OK')

## 1. Apprendre les dictionnaires dunun & jembé (pistes isolées)

On apprend sur les prises **1 et 2** (on gardera la prise 5 pour valider sans tricher).

In [ ]:
dun_files = [f'{CORP}/MJ_Woloso_{t}_D1.wav' for t in (1, 2)]
jem_files = [f'{CORP}/MJ_Woloso_{t}_{c}.wav' for t in (1, 2) for c in ('J1', 'J2')]
Wd = learn_dict(dun_files, KD)
Wj = learn_dict(jem_files, KJ)
W = np.hstack([Wd, Wj])
print('dictionnaire dunun :', Wd.shape, '| jembé :', Wj.shape, '| total :', W.shape)

## 2. Voir les dictionnaires spectraux
Le dunun = énergie **grave/concentrée** ; le jembé = **large bande**. C'est cette différence de *forme spectrale* (pas juste de fréquence) qui permet de les séparer.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True)
for k in range(KD):
    ax[0].semilogx(freqs[1:], Wd[1:, k]/(Wd[1:, k].max()+1e-9) + k, color='#2b6cff')
ax[0].set(title=f'Dictionnaire DUNUN ({KD} briques)', xlabel='Hz (log)', ylabel='brique')
for k in range(KJ):
    ax[1].semilogx(freqs[1:], Wj[1:, k]/(Wj[1:, k].max()+1e-9) + k, color='#d63bd6')
ax[1].set(title=f'Dictionnaire JEMBÉ ({KJ} briques)', xlabel='Hz (log)')
plt.tight_layout(); plt.show()

## 3. Valider sur un mix synthétique (prise 5, jamais vue)

On somme les pistes isolées de la prise 5 → un mix dont on connaît les **vrais onsets de dunun**. On compare la NMF informée à un simple filtre passe-bande.

In [ ]:
def band_onsets(y, lo=40, hi=150, delta=0.06):
    yb = sosfiltfilt(butter(4, [lo, hi], btype='band', fs=SR, output='sos'), y)
    env = librosa.onset.onset_strength(y=yb, sr=SR)
    fr = librosa.onset.onset_detect(onset_envelope=env, sr=SR, units='frames',
                                    delta=delta, wait=5, backtrack=False)
    return librosa.frames_to_time(fr, sr=SR)

parts = [load(f'{CORP}/MJ_Woloso_5_D1.wav'), load(f'{CORP}/MJ_Woloso_5_J1.wav')]
if os.path.exists(f'{CORP}/MJ_Woloso_5_J2.wav'):
    parts.append(load(f'{CORP}/MJ_Woloso_5_J2.wav'))
nn = min(map(len, parts)); mix = sum(p[:nn] for p in parts)
gt = pd.read_csv(f'{CORP}/annotations/MJ_Woloso_5_Onsets_Raw_Dundun1.csv')['Time'].values

Hm = activations(spec(mix), W)
dun_act = Hm[:KD].sum(0).astype(np.float32)
fr = librosa.onset.onset_detect(onset_envelope=dun_act, sr=SR, hop_length=HOP,
                                units='time', delta=0.15, wait=4, backtrack=False)
print(f'Vrais onsets de dunun (prise 5) : {len(gt)}\n')
P,R,F = fmeasure(band_onsets(mix), gt); print(f'  Filtre passe-bande : F={F:.2f}  (P={P:.2f} R={R:.2f})')
P,R,F = fmeasure(fr, gt);              print(f'  NMF informée       : F={F:.2f}  (P={P:.2f} R={R:.2f})')

## 4. Appliquer au Woloso YOUTUBE & séparer les voix
On garde le **même dictionnaire** (appris sur le corpus) et on résout les activations sur le mix YouTube.

In [ ]:
y = load('audio/Woloso.wav')
D = librosa.stft(y, n_fft=NFFT, hop_length=HOP)   # complexe (pour ré-écouter)
S = np.abs(D)
H = activations(S, W)
dun = H[:KD].sum(0).astype(np.float32)
jem = H[KD:].sum(0).astype(np.float32)
t = librosa.frames_to_time(np.arange(len(dun)), sr=SR, hop_length=HOP)

don = librosa.onset.onset_detect(onset_envelope=dun, sr=SR, hop_length=HOP, units='time', delta=0.15, wait=4)
jon = librosa.onset.onset_detect(onset_envelope=jem, sr=SR, hop_length=HOP, units='time', delta=0.15, wait=4)
dur = len(y)/SR
print(f'YouTube Woloso ({dur:.0f}s) :  dunun {len(don)} onsets (~{len(don)/dur:.1f}/s)  |  jembé {len(jon)} (~{len(jon)/dur:.1f}/s)')

m = (t>=30)&(t<=42)
plt.figure(figsize=(14,4))
plt.plot(t[m], dun[m]/dun.max(), color='#2b6cff', label='dunun', lw=1.3)
plt.plot(t[m], jem[m]/jem.max(), color='#d63bd6', label='jembé', alpha=.7, lw=1.0)
plt.legend(); plt.xlabel('temps (s)'); plt.title('Woloso YouTube — activités séparées (30–42s)')
plt.tight_layout(); plt.savefig(f'{OUT}/nmf_informed_youtube.png', dpi=140); plt.show()

## 5. Écouter les stems séparés (le vrai test)
On reconstruit l'audio dunun-seul et jembé-seul par masque doux. Le dunun doit sonner grave et régulier ; le jembé, plus aigu et dense.

In [ ]:
WH = W @ H + 1e-9
def stem(idx):
    mask = (W[:, idx] @ H[idx, :]) / WH
    return librosa.istft(mask * D, hop_length=HOP, length=len(y))
dun_stem = stem(list(range(KD)))
jem_stem = stem(list(range(KD, KD+KJ)))
ex = slice(int(30*SR), int(45*SR))   # extrait 30-45s
print('>>> stem DUNUN (30-45s)');  display(Audio(dun_stem[ex], rate=SR))
print('>>> stem JEMBÉ (30-45s)');  display(Audio(jem_stem[ex], rate=SR))
print('>>> mix original (30-45s)'); display(Audio(y[ex], rate=SR))

## 6. Carte des événements par instrument (YouTube)
Chaque frappe détectée, placée par son centroïde spectral, colorée par l'instrument qui l'a produite.

In [ ]:
def centroid_at(time):
    f = int(time*SR/HOP)
    if f >= S.shape[1]: return 0
    return float((S[:, f]*freqs).sum()/(S[:, f].sum()+1e-9))

ev = pd.DataFrame(
    [{'instrument':'dunun','time':x,'centroid':centroid_at(x)} for x in don] +
    [{'instrument':'jembe','time':x,'centroid':centroid_at(x)} for x in jon])
COL = {'dunun':'#2b6cff','jembe':'#d63bd6'}
plt.figure(figsize=(14,4))
for ins,c in COL.items():
    mm = ev['instrument']==ins
    plt.scatter(ev.loc[mm,'time'], ev.loc[mm,'centroid'], s=14, c=c, alpha=.6, label=ins)
plt.legend(); plt.xlabel('temps (s)'); plt.ylabel('centroïde (Hz)'); plt.xlim(30,55)
plt.title('Woloso YouTube — événements par instrument (NMF informée)'); plt.show()

## 7. Sauvegarde & suite

- On enregistre les onsets séparés par instrument.
- **Généraliser** : appliquer ce même dictionnaire aux 6 autres rythmes YouTube → carte d'instruments sur tout le corpus vidéo.
- **Phase 2** : prendre le **stem jembé** (ou la piste J1 isolée du corpus) et y étudier bass / tone / slap.
- **Phase 3** : croiser les onsets dunun/jembé avec la grille métrique (`Onsets_Selected`) pour analyser le cycle du rythme.

> Honnêteté : la validation F≈0,99 est sur mix **synthétique** (pistes propres sommées). Sur le vrai YouTube on n'a pas de vérité terrain, mais le pouls de dunun obtenu est crédible et ré-écoutable. Pour pousser, on pourrait ré-estimer les dictionnaires sur quelques secondes de chaque vidéo.

In [ ]:
ev.sort_values('time').to_csv(f'{OUT}/youtube_woloso_instruments.csv', index=False)
print('Sauvegardé ->', f'{OUT}/youtube_woloso_instruments.csv', '|', len(ev), 'événements')